In [11]:
import numpy as np
import pandas as pd

df = pd.read_csv("../aruba_cleaned.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

remove = ["Respirate", "Leave_Home", "Enter_Home", "O Relax", "OF Meal_Preparation",
          "CLOSE Enter_Home", "CLOSE Leave_Home", "OPEN Enter_Home", "OPEN Leave_Home"]
df = df[~df["activity_label"].isin(remove)]

# pair begin/end events into segments
ev = df[df["activity_label"].notna() & df["activity_state"].isin(["begin", "end"])]
ev = ev.sort_values("timestamp")
segs = []
for act, g in ev.groupby("activity_label"):
    start = None
    for ts, st in zip(g["timestamp"], g["activity_state"]):
        if st == "begin":
            start = ts
        elif start is not None:
            segs.append((act, start, ts))
            start = None
seg = pd.DataFrame(segs, columns=["activity", "start", "end"])

# split segments at midnight
rows = []
for act, s, e in seg.itertuples(index=False):
    while s.normalize() < e.normalize():
        nxt = s.normalize() + pd.Timedelta(days=1)
        rows.append((act, s, nxt))
        s = nxt
    rows.append((act, s, e))
pieces = pd.DataFrame(rows, columns=["activity", "start", "end"])
pieces["date"] = pieces["start"].dt.normalize()
pieces["minutes"] = (pieces["end"] - pieces["start"]).dt.total_seconds() / 60

# one row per day, one column per activity, 0 if absent
daily = pieces.pivot_table(index="date", columns="activity", values="minutes",
                           aggfunc="sum", fill_value=0)
daily = daily.reindex(pd.date_range(daily.index.min(), daily.index.max()), fill_value=0)
pct = daily / 1440 * 100

print("pct shape:", pct.shape)          # expect (220, 8)
print("max daily sum:", round(pct.sum(axis=1).max(), 1))

pct shape: (220, 8)
max daily sum: 96.0


In [12]:
df = pd.read_csv("../aruba_cleaned.csv")
df

,date,time,sensor,value,timestamp,sensor_type,hour,time_period,activity_label,activity_state
0,2010-11-04,00:03:50.209589,M003,ON Sleeping begin,2010-11-04 00:03:50.209589,M,0,night,Sleeping,begin
1,2010-11-04,00:03:57.399391,M003,OFF,2010-11-04 00:03:57.399391,M,0,night,NaN,NaN
2,2010-11-04,02:32:33.351906,M003,ON,2010-11-04 02:32:33.351906,M,2,night,NaN,NaN
3,2010-11-04,02:32:38.895958,M003,OFF,2010-11-04 02:32:38.895958,M,2,night,NaN,NaN
4,2010-11-04,03:42:21.82365,M003,ON,2010-11-04 03:42:21.823650,M,3,night,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1602976,2011-06-11,22:33:01.209214,M002,ON,2011-06-11 22:33:01.209214,M,22,night,NaN,NaN
1602977,2011-06-11,22:33:02.952009,M003,OFF,2011-06-11 22:33:02.952009,M,22,night,NaN,NaN
1602978,2011-06-11,22:33:03.332876,M002,OFF,2011-06-11 22:33:03.332876,M,22,night,NaN,NaN
1602979,2011-06-11,23:20:31.432636,M003,ON,2011-06-11 23:20:31.432636,M,23,night,NaN,NaN


In [13]:
activities = df["activity_label"].dropna().unique()

print(activities)

['Sleeping' 'Bed_to_Toilet' 'Meal_Preparation' 'Relax' 'Housekeeping'
 'Eating' 'Wash_Dishes' 'OPEN Leave_Home' 'CLOSE Leave_Home'
 'OPEN Enter_Home' 'CLOSE Enter_Home' 'Work' 'Respirate' 'Leave_Home'
 'Enter_Home' 'O Relax' 'OF Meal_Preparation']


In [14]:
# Make sure timestamp is datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Keep only rows that have an activity
activity_df = df[df["activity_label"].notna()].copy()

# Sort by time
activity_df = activity_df.sort_values("timestamp")

activities = []

for activity, group in activity_df.groupby("activity_label"):
    
    group = group.sort_values("timestamp")
    start_time = None

    for _, row in group.iterrows():

        if row["activity_state"] == "begin":
            start_time = row["timestamp"]

        elif row["activity_state"] == "end" and start_time is not None:

            end_time = row["timestamp"]

            duration = end_time - start_time

            activities.append({
                "activity": activity,
                "start": start_time,
                "end": end_time,
                "duration": duration
            })

            start_time = None

# Create DataFrame
activity_duration_df = pd.DataFrame(activities)

# Convert duration to minutes
activity_duration_df["duration_minutes"] = (
    activity_duration_df["duration"].dt.total_seconds() / 60
)

print(activity_duration_df.head())

        activity                      start                        end  \
0  Bed_to_Toilet 2010-11-04 05:40:51.303739 2010-11-04 05:43:30.279021   
1  Bed_to_Toilet 2010-11-05 04:18:09.406239 2010-11-05 04:20:50.109684   
2  Bed_to_Toilet 2010-11-06 04:23:50.544193 2010-11-06 04:27:00.048965   
3  Bed_to_Toilet 2010-11-07 00:34:48.871659 2010-11-07 00:35:36.223000   
4  Bed_to_Toilet 2010-11-07 03:43:19.802838 2010-11-07 03:46:23.886632   

                duration  duration_minutes  
0 0 days 00:02:38.975282          2.649588  
1 0 days 00:02:40.703445          2.678391  
2 0 days 00:03:09.504772          3.158413  
3 0 days 00:00:47.351341          0.789189  
4 0 days 00:03:04.083794          3.068063  


In [15]:
duration_summary = (
    activity_duration_df
    .groupby("activity")["duration_minutes"]
    .agg(
        average_duration="mean",
        min_duration="min",
        max_duration="max"
    )
    .reset_index()
)

print(duration_summary)

            activity  average_duration  min_duration  max_duration
0      Bed_to_Toilet          2.729481      0.357222      8.702630
1   CLOSE Enter_Home        829.721475    829.721475    829.721475
2   CLOSE Leave_Home        431.749579    431.749579    431.749579
3             Eating         10.149268      0.264641     69.475333
4         Enter_Home        391.872700    391.872700    391.872700
5       Housekeeping         20.324018      1.536077     90.010639
6         Leave_Home       7846.165769      0.014001  15692.317537
7   Meal_Preparation          7.838226      0.126168    326.200545
8              Relax         33.431752      0.944290    255.024874
9          Respirate          8.564308      1.638024     24.919531
10          Sleeping        241.214425      0.555374    646.396949
11       Wash_Dishes          7.246125      0.554909     36.654173
12              Work         17.080463      0.913070    116.940938


In [16]:
activities_to_remove = [
    "Respirate",
    "Leave_Home",
    "Enter_Home",
    "O Relax",
    "OF Meal_Preparation"
]

df = df[~df["activity_label"].isin(activities_to_remove)].copy()

In [17]:
activity_counts = df["activity_label"].value_counts()

print(activity_counts)

activity_label
Relax               5837
Meal_Preparation    3211
Sleeping             801
Eating               514
CLOSE Enter_Home     431
OPEN Leave_Home      427
CLOSE Leave_Home     426
OPEN Enter_Home      425
Work                 342
Bed_to_Toilet        314
Wash_Dishes          130
Housekeeping          66
Name: count, dtype: int64


In [18]:
duration_summary = (
    activity_duration_df
    .groupby("activity")["duration_minutes"]
    .agg(
        average_duration="mean",
        min_duration="min",
        max_duration="max"
    )
    .reset_index()
)

print(duration_summary)

            activity  average_duration  min_duration  max_duration
0      Bed_to_Toilet          2.729481      0.357222      8.702630
1   CLOSE Enter_Home        829.721475    829.721475    829.721475
2   CLOSE Leave_Home        431.749579    431.749579    431.749579
3             Eating         10.149268      0.264641     69.475333
4         Enter_Home        391.872700    391.872700    391.872700
5       Housekeeping         20.324018      1.536077     90.010639
6         Leave_Home       7846.165769      0.014001  15692.317537
7   Meal_Preparation          7.838226      0.126168    326.200545
8              Relax         33.431752      0.944290    255.024874
9          Respirate          8.564308      1.638024     24.919531
10          Sleeping        241.214425      0.555374    646.396949
11       Wash_Dishes          7.246125      0.554909     36.654173
12              Work         17.080463      0.913070    116.940938


In [19]:
activity_duration_df["date"] = activity_duration_df["start"].dt.date

activity_duration_df["hour"] = activity_duration_df["start"].dt.hour

activity_duration_df["minute"] = activity_duration_df["start"].dt.minute

activity_duration_df["day_of_week"] = (
    activity_duration_df["start"].dt.day_name()
)

activity_duration_df["start_time"] = (
    activity_duration_df["start"].dt.time
)

print(activity_duration_df.head(50))

         activity                      start                        end  \
0   Bed_to_Toilet 2010-11-04 05:40:51.303739 2010-11-04 05:43:30.279021   
1   Bed_to_Toilet 2010-11-05 04:18:09.406239 2010-11-05 04:20:50.109684   
2   Bed_to_Toilet 2010-11-06 04:23:50.544193 2010-11-06 04:27:00.048965   
3   Bed_to_Toilet 2010-11-07 00:34:48.871659 2010-11-07 00:35:36.223000   
4   Bed_to_Toilet 2010-11-07 03:43:19.802838 2010-11-07 03:46:23.886632   
5   Bed_to_Toilet 2010-11-07 03:49:20.647413 2010-11-07 03:51:17.964123   
6   Bed_to_Toilet 2010-11-08 05:40:10.556874 2010-11-08 05:43:15.104099   
7   Bed_to_Toilet 2010-11-09 00:30:32.314798 2010-11-09 00:32:54.868485   
8   Bed_to_Toilet 2010-11-09 05:31:30.888221 2010-11-09 05:34:08.785188   
9   Bed_to_Toilet 2010-11-11 05:13:58.821109 2010-11-11 05:16:28.261014   
10  Bed_to_Toilet 2010-11-12 04:31:41.818859 2010-11-12 04:35:07.429031   
11  Bed_to_Toilet 2010-11-13 05:17:05.045679 2010-11-13 05:20:52.478511   
12  Bed_to_Toilet 2010-11

In [20]:
routine_summary = (
    activity_duration_df
    .groupby("activity")
    .agg(
        average_start_hour=("hour", "mean"),
        average_duration=("duration_minutes", "mean"),
        min_duration=("duration_minutes", "min"),
        max_duration=("duration_minutes", "max"),
        frequency=("activity", "count") #el frequency of each activity yaane kam event et3ml w kheles completed
    )
    .reset_index()
)

print(routine_summary)

            activity  average_start_hour  average_duration  min_duration  \
0      Bed_to_Toilet            4.192308          2.729481      0.357222   
1   CLOSE Enter_Home           20.000000        829.721475    829.721475   
2   CLOSE Leave_Home            8.000000        431.749579    431.749579   
3             Eating           12.670588         10.149268      0.264641   
4         Enter_Home           14.000000        391.872700    391.872700   
5       Housekeeping           12.575758         20.324018      1.536077   
6         Leave_Home           13.500000       7846.165769      0.014001   
7   Meal_Preparation           12.578056          7.838226      0.126168   
8              Relax           15.271851         33.431752      0.944290   
9          Respirate           11.166667          8.564308      1.638024   
10          Sleeping            8.801008        241.214425      0.555374   
11       Wash_Dishes           14.937500          7.246125      0.554909   
12          

In [21]:
routine_data = activity_duration_df[
    [
        "date",
        "activity",
        "start",
        "duration_minutes",
        "day_of_week"
    ]
].copy()

print(routine_data.head(20))

          date       activity                      start  duration_minutes  \
0   2010-11-04  Bed_to_Toilet 2010-11-04 05:40:51.303739          2.649588   
1   2010-11-05  Bed_to_Toilet 2010-11-05 04:18:09.406239          2.678391   
2   2010-11-06  Bed_to_Toilet 2010-11-06 04:23:50.544193          3.158413   
3   2010-11-07  Bed_to_Toilet 2010-11-07 00:34:48.871659          0.789189   
4   2010-11-07  Bed_to_Toilet 2010-11-07 03:43:19.802838          3.068063   
5   2010-11-07  Bed_to_Toilet 2010-11-07 03:49:20.647413          1.955278   
6   2010-11-08  Bed_to_Toilet 2010-11-08 05:40:10.556874          3.075787   
7   2010-11-09  Bed_to_Toilet 2010-11-09 00:30:32.314798          2.375895   
8   2010-11-09  Bed_to_Toilet 2010-11-09 05:31:30.888221          2.631616   
9   2010-11-11  Bed_to_Toilet 2010-11-11 05:13:58.821109          2.490665   
10  2010-11-12  Bed_to_Toilet 2010-11-12 04:31:41.818859          3.426836   
11  2010-11-13  Bed_to_Toilet 2010-11-13 05:17:05.045679        

In [22]:
daily_frequency = (
    routine_data
    .groupby(["date", "activity"])
    .size()
    .reset_index(name="count")
)

print(daily_frequency.tail())

            date          activity  count
1088  2011-06-10              Work      1
1089  2011-06-11            Eating      1
1090  2011-06-11  Meal_Preparation      8
1091  2011-06-11             Relax     17
1092  2011-06-11              Work      1


In [23]:
average_frequency = (
    daily_frequency
    .groupby("activity")["count"]
    .mean()
    .reset_index(name="average_daily_frequency")
)

print(average_frequency)

            activity  average_daily_frequency
0      Bed_to_Toilet                 1.172932
1   CLOSE Enter_Home                 1.000000
2   CLOSE Leave_Home                 1.000000
3             Eating                 1.861314
4         Enter_Home                 1.000000
5       Housekeeping                 1.178571
6         Leave_Home                 1.000000
7   Meal_Preparation                 7.283105
8              Relax                13.269406
9          Respirate                 1.000000
10          Sleeping                 1.965347
11       Wash_Dishes                 1.254902
12              Work                 1.838710


In [24]:
activities_to_remove = [
    "CLOSE Enter_Home",
    "CLOSE Leave_Home",
    "OPEN Enter_Home",
    "OPEN Leave_Home",
    "Enter_Home",
    "Leave_Home",
    "Respirate"
] #shelna el door activities for now

df = df[~df["activity_label"].isin(activities_to_remove)].copy()

print(df["activity_label"].value_counts())

activity_label
Relax               5837
Meal_Preparation    3211
Sleeping             801
Eating               514
Work                 342
Bed_to_Toilet        314
Wash_Dishes          130
Housekeeping          66
Name: count, dtype: int64


In [25]:
# Activities we don't want in the routine baseline
activities_to_remove = [
    "CLOSE Enter_Home",
    "CLOSE Leave_Home",
    "OPEN Enter_Home",
    "OPEN Leave_Home",
    "Enter_Home",
    "Leave_Home",
    "Respirate"
]

# Remove them from df
df = df[~df["activity_label"].isin(activities_to_remove)].copy()


# =========================================
# Recreate activity_duration_df
# =========================================

df["timestamp"] = pd.to_datetime(df["timestamp"])

activity_df = df[df["activity_label"].notna()].copy()
activity_df = activity_df.sort_values("timestamp")

activities = []

for activity, group in activity_df.groupby("activity_label"):

    group = group.sort_values("timestamp")
    start_time = None

    for _, row in group.iterrows():

        if row["activity_state"] == "begin":
            start_time = row["timestamp"]

        elif row["activity_state"] == "end" and start_time is not None:

            end_time = row["timestamp"]

            activities.append({
                "activity": activity,
                "start": start_time,
                "end": end_time,
                "duration": end_time - start_time
            })

            start_time = None


activity_duration_df = pd.DataFrame(activities)

# Duration in minutes
activity_duration_df["duration_minutes"] = (
    activity_duration_df["duration"].dt.total_seconds() / 60
)

# Add time features
activity_duration_df["date"] = activity_duration_df["start"].dt.date
activity_duration_df["hour"] = activity_duration_df["start"].dt.hour
activity_duration_df["minute"] = activity_duration_df["start"].dt.minute

In [26]:
#baseline table
daily_frequency = (
    activity_duration_df
    .groupby(["date", "activity"])
    .size()
    .reset_index(name="daily_frequency")
)

average_frequency = (
    daily_frequency
    .groupby("activity")["daily_frequency"]
    .mean()
    .reset_index(name="average_daily_frequency")
)

duration_stats = (
    activity_duration_df
    .groupby("activity")
    .agg(
        average_duration=("duration_minutes", "mean"),
        min_duration=("duration_minutes", "min"),
        max_duration=("duration_minutes", "max"),
        average_start_hour=("hour", "mean")
    )
    .reset_index()
)

baseline_table = average_frequency.merge(
    duration_stats,
    on="activity"
)

baseline_table = baseline_table.round(2)

print(baseline_table)

           activity  average_daily_frequency  average_duration  min_duration  \
0     Bed_to_Toilet                     1.17              2.73          0.36   
1            Eating                     1.86             10.15          0.26   
2      Housekeeping                     1.18             20.32          1.54   
3  Meal_Preparation                     7.28              7.84          0.13   
4             Relax                    13.27             33.43          0.94   
5          Sleeping                     1.97            241.21          0.56   
6       Wash_Dishes                     1.25              7.25          0.55   
7              Work                     1.84             17.08          0.91   

   max_duration  average_start_hour  
0          8.70                4.19  
1         69.48               12.67  
2         90.01               12.58  
3        326.20               12.58  
4        255.02               15.27  
5        646.40                8.80  
6         36.

In [27]:
# =========================================
# Daily Routine Table
# =========================================
#What is normal for this person overall


daily_routine = (
    activity_duration_df
    .groupby(["date", "activity"])
    .agg(
        frequency=("activity", "count"),
        total_duration=("duration_minutes", "sum"),
        average_duration=("duration_minutes", "mean"),
        average_start_hour=("hour", "mean")
    )
    .reset_index()
)

# Round the numbers
daily_routine = daily_routine.round(2)

print(daily_routine.head(20))

          date          activity  frequency  total_duration  average_duration  \
0   2010-11-04     Bed_to_Toilet          1            2.65              2.65   
1   2010-11-04            Eating          4           10.44              2.61   
2   2010-11-04      Housekeeping          2           15.06              7.53   
3   2010-11-04  Meal_Preparation          9           67.29              7.48   
4   2010-11-04             Relax          5           64.01             12.80   
5   2010-11-04          Sleeping          2          474.33            237.17   
6   2010-11-04       Wash_Dishes          2           19.00              9.50   
7   2010-11-04              Work          2           25.13             12.57   
8   2010-11-05     Bed_to_Toilet          1            2.68              2.68   
9   2010-11-05            Eating          1            3.50              3.50   
10  2010-11-05      Housekeeping          1           90.01             90.01   
11  2010-11-05  Meal_Prepara

In [28]:
daily_routine["activity_time_percentage"] = (
    daily_routine["total_duration"] / 1440
) * 100
daily_routine["activity_time_percentage"] = (
    daily_routine["activity_time_percentage"].round(2)
)


In [29]:
daily_routine.head(50) #dedication yaane el activity de akhadet kan fl meya mn youmo

,date,activity,frequency,total_duration,average_duration,average_start_hour,activity_time_percentage
0,2010-11-04,Bed_to_Toilet,1,2.65,2.65,5.00,0.18
1,2010-11-04,Eating,4,10.44,2.61,12.50,0.72
2,2010-11-04,Housekeeping,2,15.06,7.53,10.00,1.05
3,2010-11-04,Meal_Preparation,9,67.29,7.48,12.67,4.67
4,2010-11-04,Relax,5,64.01,12.80,14.60,4.45
5,2010-11-04,Sleeping,2,474.33,237.17,2.50,32.94
6,2010-11-04,Wash_Dishes,2,19.00,9.50,13.50,1.32
7,2010-11-04,Work,2,25.13,12.57,16.00,1.75
8,2010-11-05,Bed_to_Toilet,1,2.68,2.68,4.00,0.19
9,2010-11-05,Eating,1,3.50,3.50,16.00,0.24


In [30]:
# =========================================
# Compare daily routine with baseline
# =========================================

routine_comparison = daily_routine.merge(
    baseline_table,
    on="activity",
    suffixes=("_daily", "_baseline")
)

print(routine_comparison.tail(20))

            date          activity  frequency  total_duration  \
1062  2011-06-07          Sleeping          2          388.61   
1063  2011-06-08     Bed_to_Toilet          1            2.92   
1064  2011-06-08            Eating          2           16.81   
1065  2011-06-08  Meal_Preparation         10           52.41   
1066  2011-06-08             Relax         15          401.77   
1067  2011-06-08          Sleeping          2          410.74   
1068  2011-06-08              Work          2            7.48   
1069  2011-06-09     Bed_to_Toilet          1            4.64   
1070  2011-06-09  Meal_Preparation          5           22.56   
1071  2011-06-09             Relax          7          285.37   
1072  2011-06-09          Sleeping          2          526.11   
1073  2011-06-10            Eating          2           17.67   
1074  2011-06-10  Meal_Preparation         11           77.52   
1075  2011-06-10             Relax          8          314.07   
1076  2011-06-10         

In [31]:
#mmkn neshel mn awl hena 
# Frequency deviation
routine_comparison["frequency_deviation"] = (
    routine_comparison["frequency"]
    - routine_comparison["average_daily_frequency"]
)

# Duration deviation
routine_comparison["duration_deviation"] = (
    routine_comparison["average_duration_daily"]
    - routine_comparison["average_duration_baseline"]
)

# Start-time deviation
routine_comparison["start_time_deviation"] = (
    routine_comparison["average_start_hour_daily"]
    - routine_comparison["average_start_hour_baseline"]
)

# Round the results
routine_comparison[
    [
        "frequency_deviation",
        "duration_deviation",
        "start_time_deviation"
    ]
] = routine_comparison[
    [
        "frequency_deviation",
        "duration_deviation",
        "start_time_deviation"
    ]
].round(2)

print(routine_comparison.head(20))

          date          activity  frequency  total_duration  \
0   2010-11-04     Bed_to_Toilet          1            2.65   
1   2010-11-04            Eating          4           10.44   
2   2010-11-04      Housekeeping          2           15.06   
3   2010-11-04  Meal_Preparation          9           67.29   
4   2010-11-04             Relax          5           64.01   
5   2010-11-04          Sleeping          2          474.33   
6   2010-11-04       Wash_Dishes          2           19.00   
7   2010-11-04              Work          2           25.13   
8   2010-11-05     Bed_to_Toilet          1            2.68   
9   2010-11-05            Eating          1            3.50   
10  2010-11-05      Housekeeping          1           90.01   
11  2010-11-05  Meal_Preparation          3            8.02   
12  2010-11-05             Relax          2            9.76   
13  2010-11-05          Sleeping          3          800.32   
14  2010-11-05       Wash_Dishes          1            

In [32]:
routine_comparison = daily_routine.merge(
    baseline_table,
    on="activity",
    suffixes=("_daily", "_baseline")
)

routine_comparison["frequency_deviation"] = (
    routine_comparison["frequency"]
    - routine_comparison["average_daily_frequency"]
)

routine_comparison["duration_deviation"] = (
    routine_comparison["average_duration_daily"]
    - routine_comparison["average_duration_baseline"]
)

routine_comparison["start_time_deviation"] = (
    routine_comparison["average_start_hour_daily"]
    - routine_comparison["average_start_hour_baseline"]
)

In [33]:
routine_comparison.columns

Index(['date', 'activity', 'frequency', 'total_duration',
       'average_duration_daily', 'average_start_hour_daily',
       'activity_time_percentage', 'average_daily_frequency',
       'average_duration_baseline', 'min_duration', 'max_duration',
       'average_start_hour_baseline', 'frequency_deviation',
       'duration_deviation', 'start_time_deviation'],
      dtype='object')

In [34]:
variability = (
    daily_routine
    .groupby("activity")
    .agg(
        frequency_std=("frequency", "std"),
        duration_std=("average_duration", "std"),
        start_time_std=("average_start_hour", "std")
    )
    .reset_index()
)

variability

,activity,frequency_std,duration_std,start_time_std
0,Bed_to_Toilet,0.435390,0.985498,2.613255
1,Eating,1.356805,7.713036,3.528412
2,Housekeeping,0.390021,20.218954,2.987186
3,Meal_Preparation,3.554586,4.260761,2.147345
4,Relax,4.000634,11.306951,1.645044
5,Sleeping,0.883113,100.152102,6.712690
6,Wash_Dishes,0.483452,7.489216,4.328433
7,Work,1.236143,18.356466,3.579488


In [35]:
routine_comparison = routine_comparison.merge(
    variability,
    on="activity",
    how="left"
)

In [36]:
import numpy as np

routine_comparison["frequency_deviation_score"] = (
    routine_comparison["frequency_deviation"].abs()
    / routine_comparison["frequency_std"]
)

routine_comparison["duration_deviation_score"] = (
    routine_comparison["duration_deviation"].abs()
    / routine_comparison["duration_std"]
)

routine_comparison["start_time_deviation_score"] = (
    routine_comparison["start_time_deviation"].abs()
    / routine_comparison["start_time_std"]
)

In [37]:
score_columns = [
    "frequency_deviation_score",
    "duration_deviation_score",
    "start_time_deviation_score"
]

routine_comparison[score_columns] = (
    routine_comparison[score_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .round(2)
)

In [38]:
routine_comparison[
    [
        "date",
        "activity",
        "frequency_deviation_score",
        "duration_deviation_score",
        "start_time_deviation_score"
    ]
].head(20)

,date,activity,frequency_deviation_score,duration_deviation_score,start_time_deviation_score
0,2010-11-04,Bed_to_Toilet,0.39,0.08,0.31
1,2010-11-04,Eating,1.58,0.98,0.05
2,2010-11-04,Housekeeping,2.10,0.63,0.86
3,2010-11-04,Meal_Preparation,0.48,0.08,0.04
4,2010-11-04,Relax,2.07,1.82,0.41
5,2010-11-04,Sleeping,0.03,0.04,0.94
6,2010-11-04,Wash_Dishes,1.55,0.30,0.33
7,2010-11-04,Work,0.13,0.25,0.85
8,2010-11-05,Bed_to_Toilet,0.39,0.05,0.07
9,2010-11-05,Eating,0.63,0.86,0.94


In [39]:
def build_segments(df):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    ev = (df[df["activity_label"].notna() & df["activity_state"].isin(["begin", "end"])]
            .sort_values("timestamp"))
    segs = []
    for act, g in ev.groupby("activity_label"):
        start = None
        for ts, st in zip(g["timestamp"], g["activity_state"]):
            if st == "begin":
                start = ts
            elif st == "end" and start is not None:
                segs.append((act, start, ts))
                start = None
    return pd.DataFrame(segs, columns=["activity", "start", "end"])

def split_midnight(seg):
    rows = []
    for act, s, e in seg.itertuples(index=False):
        while s.normalize() < e.normalize():
            nxt = s.normalize() + pd.Timedelta(days=1)
            rows.append((act, s, nxt))
            s = nxt
        rows.append((act, s, e))
    out = pd.DataFrame(rows, columns=["activity", "start", "end"])
    out["date"] = out["start"].dt.normalize()
    out["minutes"] = (out["end"] - out["start"]).dt.total_seconds() / 60
    return out

drop = ["CLOSE Enter_Home","CLOSE Leave_Home","OPEN Enter_Home","OPEN Leave_Home",
        "Enter_Home","Leave_Home","Respirate","O Relax","OF Meal_Preparation"]
df = df[~df["activity_label"].isin(drop)]
seg = split_midnight(build_segments(df))
print(seg.sort_values("minutes", ascending=False).head(10))   # check outliers

      activity                      start                        end  \
5157  Sleeping 2010-12-21 00:36:30.072849 2010-12-21 10:25:15.063739   
5173  Sleeping 2010-12-27 00:00:00.000000 2010-12-27 08:57:39.329977   
5104  Sleeping 2010-12-01 00:00:00.000000 2010-12-01 08:51:28.370488   
5187  Sleeping 2011-01-03 00:22:30.081303 2011-01-03 08:40:16.401941   
5205  Sleeping 2011-01-12 00:07:13.812782 2011-01-12 08:19:22.348419   
5062  Sleeping 2010-11-16 00:39:27.333743 2010-11-16 08:48:00.604924   
5326  Sleeping 2011-03-17 00:02:43.090986 2011-03-17 08:09:30.402476   
5228  Sleeping 2011-01-24 00:00:00.000000 2011-01-24 07:55:11.084163   
5283  Sleeping 2011-02-25 00:42:36.785365 2011-02-25 08:36:41.586419   
5214  Sleeping 2011-01-16 00:41:22.538723 2011-01-16 08:34:24.199291   

           date     minutes  
5157 2010-12-21  588.749848  
5173 2010-12-27  537.655500  
5104 2010-12-01  531.472841  
5187 2011-01-03  497.772011  
5205 2011-01-12  492.142261  
5062 2010-11-16  488.554520

In [40]:
daily = seg.pivot_table(index="date", columns="activity",
                        values="minutes", aggfunc="sum", fill_value=0)
full = pd.date_range(daily.index.min(), daily.index.max())
daily = daily.reindex(full)                 # missing days become NaN, not 0
print("missing days:", daily.isna().all(axis=1).sum())
pct = daily / 1440 * 100                    # compute before any rounding
print(pct.sum(axis=1).describe())           # should be <= 100

missing days: 0
count    220.000000
mean      67.072549
std       10.532479
min       30.489338
25%       60.970070
50%       67.133183
75%       73.940266
max       96.005798
dtype: float64


In [41]:
split = int(len(pct) * 0.7)
train, test = pct.iloc[:split], pct.iloc[split:]

def make_windows(d, w=15):
    v = d.values
    X = np.array([v[i:i+w-1] for i in range(len(v)-w+1)])
    y = np.array([v[i+w-1]   for i in range(len(v)-w+1)])
    return X, y

Xtr, ytr = make_windows(train)
Xte, yte = make_windows(test)
print(Xtr.shape, Xte.shape)    

(140, 14, 8) (52, 14, 8)


In [42]:
W = 15
split = int(len(pct) * 0.7)                    # first 70% of days = train
train, test = pct.iloc[:split], pct.iloc[split:]

def make_windows(d, w=W):
    v = d.values
    X = np.array([v[i:i+w-1] for i in range(len(v)-w+1)])
    y = np.array([v[i+w-1] for i in range(len(v)-w+1)])
    return X, y

Xtr, ytr = make_windows(train)
Xte, yte = make_windows(test)
print("train:", Xtr.shape, "test:", Xte.shape)   # expect (140, 14, 8) and (52, 14, 8)

def mse(p, y): return np.mean((p - y) ** 2)
def mae(p, y): return np.mean(np.abs(p - y))

mean_pred = np.tile(ytr.mean(axis=0), (len(yte), 1))
print("Mean baseline  MSE:", round(mse(mean_pred, yte), 2), " MAE:", round(mae(mean_pred, yte), 2))

train: (140, 14, 8) test: (52, 14, 8)
Mean baseline  MSE: 15.19  MAE: 1.92


In [43]:
!pip install tensorflow-cpu

   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/351.2 MB 407.0 kB/s eta 0:14:22
   ---------------------------------------- 0.5/351.2 MB 407.0 kB/s eta 0:14:22
   ---------------------------------------- 0.5/351.2 MB 407.0 kB/s eta 0:14:22
   ---------------------------------------- 0.5/351.2 MB 407.0 kB/s eta 0:14:22
   ---------------------------------------- 0.8/351.2 MB 340.5 kB/s eta 0:17:10
   ---------------------------------------- 0.8/351.2 MB 340.5 kB/s eta 0:17:10
   ---------------------------------------- 0.8/351.2 MB 340.5 kB/s eta 0:17:10
   ---------------------------------------- 1.0/351.2 MB 396.0 kB/s eta 0:14:45
   --


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [44]:
import tensorflow as tf
from tensorflow import keras

tf.keras.utils.set_random_seed(0)

model = keras.Sequential([
    keras.Input((W - 1, pct.shape[1])),
    keras.layers.LSTM(64, dropout=0.1),
    keras.layers.Dense(pct.shape[1], activation="relu"),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999),
              loss="mse")
model.fit(Xtr, ytr, batch_size=128, epochs=80, verbose=0)

p = model.predict(Xte, verbose=0)
print("LSTM  MSE:", round(mse(p, yte), 2), " MAE:", round(mae(p, yte), 2))

LSTM  MSE: 61.59  MAE: 4.01


In [45]:
from scipy import stats

# step 6a: z-score of each day's prediction error
err_tr = np.abs(model.predict(Xtr, verbose=0) - ytr).mean(axis=1)
err_te = np.abs(p - yte).mean(axis=1)
err = np.concatenate([err_tr, err_te])
z = (err - err.mean()) / err.std()
Z = stats.norm.ppf(0.925)                       # paper's threshold, about 1.44
flag_te = z[len(err_tr):] > Z
print("threshold Z:", round(Z, 3), "| flagged test days:", int(flag_te.sum()), "of", len(flag_te))

# step 6b: which activities are outside the boxplot limits (limits from training days only)
q1, q3 = train.quantile(0.25), train.quantile(0.75)
upper, lower = q3 + 1.5 * (q3 - q1), q1 - 1.5 * (q3 - q1)

test_dates = pct.index[split + W - 1:]          # date of each test label
rows = []
for i in np.where(flag_te)[0]:
    day = pct.loc[test_dates[i]]
    for a in pct.columns:
        if day[a] > upper[a]:
            rows.append((test_dates[i].date(), a, round(day[a], 2), "over", round(upper[a], 2)))
        elif day[a] < lower[a]:
            rows.append((test_dates[i].date(), a, round(day[a], 2), "under", round(lower[a], 2)))

anom = pd.DataFrame(rows, columns=["date", "activity", "dedication_%", "direction", "limit_%"])
print(anom.to_string())
anom.to_csv("aruba_anomalies.csv", index=False)

threshold Z: 1.44 | flagged test days: 0 of 52
Empty DataFrame
Columns: [date, activity, dedication_%, direction, limit_%]
Index: []


In [46]:
# 1. simple persistence baseline: tomorrow = today
persist_pred = np.array([pct.values[split + i + W - 2] for i in range(len(yte))])
print("Persistence MSE:", round(mse(persist_pred, yte), 2), " MAE:", round(mae(persist_pred, yte), 2))

# 2. where the LSTM error comes from, per activity
lstm_err = ((p - yte) ** 2).mean(axis=0)
base_err = ((mean_pred - yte) ** 2).mean(axis=0)
print(pd.DataFrame({"LSTM MSE": lstm_err, "Mean MSE": base_err}, index=pct.columns).round(2))

Persistence MSE: 27.07  MAE: 2.41
                  LSTM MSE  Mean MSE
activity                            
Bed_to_Toilet         0.01      0.01
Eating                0.94      0.96
Housekeeping          0.00      0.03
Meal_Preparation      9.59      9.36
Relax               239.64     85.37
Sleeping            241.19     24.40
Wash_Dishes           0.02      0.04
Work                  1.31      1.31


In [47]:
p_mean = p.mean(axis=0)
y_mean = yte.mean(axis=0)
tr_mean = ytr.mean(axis=0)
print(pd.DataFrame({
    "train mean %": tr_mean,
    "test true mean %": y_mean,
    "LSTM pred mean %": p_mean,
}, index=pct.columns).round(2))

print("train std:", np.round(ytr.std(axis=0), 2))
print("test std: ", np.round(yte.std(axis=0), 2))

                  train mean %  test true mean %  LSTM pred mean %
activity                                                          
Bed_to_Toilet             0.14              0.09          0.160000
Eating                    0.79              0.51          0.720000
Housekeeping              0.18              0.00          0.000000
Meal_Preparation          3.81              4.06          3.520000
Relax                    32.28             29.24         16.459999
Sleeping                 30.16             29.09         14.330000
Wash_Dishes               0.18              0.03          0.000000
Work                      0.84              0.78          0.820000
train std: [ 0.15  1.19  0.62  2.55 11.5   6.42  0.45  1.89]
test std:  [0.09 0.94 0.   3.05 8.73 4.82 0.12 1.14]


In [48]:
def fit_eval(lr, scale, epochs=300):
    if scale:
        mu, sd = train.values.mean(0), train.values.std(0) + 1e-8
        Xa, ya = (Xtr - mu) / sd, (ytr - mu) / sd
        Xb = (Xte - mu) / sd
    else:
        mu, sd = 0.0, 1.0
        Xa, ya, Xb = Xtr, ytr, Xte

    tf.keras.utils.set_random_seed(0)
    m = keras.Sequential([
        keras.Input((W - 1, pct.shape[1])),
        keras.layers.LSTM(64),
        keras.layers.Dense(pct.shape[1], activation="linear" if scale else "relu"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss="mse")
    h = m.fit(Xa, ya, batch_size=32, epochs=epochs, verbose=0)

    pred = m.predict(Xb, verbose=0) * sd + mu
    print(f"lr={lr:<6} scaled={scale!s:<5}  final train loss={h.history['loss'][-1]:.4f}  "
          f"test MSE={mse(pred, yte):7.2f}  MAE={mae(pred, yte):.2f}  "
          f"Relax pred={pred[:,4].mean():5.1f} (true {yte[:,4].mean():.1f})  "
          f"Sleep pred={pred[:,5].mean():5.1f} (true {yte[:,5].mean():.1f})")
    return pred

for lr in [0.001, 0.01]:
    for scale in [False, True]:
        fit_eval(lr, scale)

lr=0.001  scaled=False  final train loss=20.5191  test MSE=  14.99  MAE=1.86  Relax pred= 30.7 (true 29.2)  Sleep pred= 29.9 (true 29.1)
lr=0.001  scaled=True   final train loss=0.0033  test MSE=  36.63  MAE=3.25  Relax pred= 30.4 (true 29.2)  Sleep pred= 35.0 (true 29.1)
lr=0.01   scaled=False  final train loss=9.3153  test MSE=  26.41  MAE=2.33  Relax pred= 34.2 (true 29.2)  Sleep pred= 30.8 (true 29.1)
lr=0.01   scaled=True   final train loss=0.0006  test MSE=  25.26  MAE=2.68  Relax pred= 32.4 (true 29.2)  Sleep pred= 33.2 (true 29.1)


In [49]:
tf.keras.utils.set_random_seed(0)
model = keras.Sequential([
    keras.Input((W - 1, pct.shape[1])),
    keras.layers.LSTM(64, dropout=0.1),
    keras.layers.Dense(pct.shape[1], activation="relu"),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="mse")
model.fit(Xtr, ytr, batch_size=128, epochs=300, verbose=0)

ptr = model(Xtr, training=False).numpy()    # direct call avoids the retracing warnings
pte = model(Xte, training=False).numpy()
print("LSTM test MSE:", round(mse(pte, yte), 2), " MAE:", round(mae(pte, yte), 2))

# step 6a: z-score of each day's prediction error
err_tr = np.abs(ptr - ytr).mean(axis=1)
err_te = np.abs(pte - yte).mean(axis=1)
err = np.concatenate([err_tr, err_te])
z = (err - err.mean()) / err.std()
Z = stats.norm.ppf(0.925)
flag_te = z[len(err_tr):] > Z
print("threshold Z:", round(Z, 3), "| flagged test days:", int(flag_te.sum()), "of", len(flag_te))

# step 6b: boxplot limits from training days only
q1, q3 = train.quantile(0.25), train.quantile(0.75)
upper, lower = q3 + 1.5 * (q3 - q1), q1 - 1.5 * (q3 - q1)
test_dates = pct.index[split + W - 1:]
rows = []
for i in np.where(flag_te)[0]:
    day = pct.loc[test_dates[i]]
    for a in pct.columns:
        if day[a] > upper[a]:
            rows.append((test_dates[i].date(), a, round(day[a], 2), "over", round(upper[a], 2)))
        elif day[a] < lower[a]:
            rows.append((test_dates[i].date(), a, round(day[a], 2), "under", round(lower[a], 2)))
anom = pd.DataFrame(rows, columns=["date", "activity", "dedication_%", "direction", "limit_%"])
print(anom.to_string())
anom.to_csv("aruba_anomalies.csv", index=False)


LSTM test MSE: 14.02  MAE: 1.81
threshold Z: 1.44 | flagged test days: 3 of 52
         date          activity  dedication_% direction  limit_%
0  2011-05-22  Meal_Preparation         14.27      over     8.71
1  2011-05-30             Relax          0.00     under     2.14
2  2011-06-11  Meal_Preparation         10.56      over     8.71
